# 5.3 Jouw beurt: overlijdens volgen tests — maar hoe lang blijft dat gelden?

Elke relatie in deze les tot nu toe werd getoetst aan de hand van de vraag wat haar geloofwaardig zou maken: een mechanisme, bewijs op het niveau van de eenheid van de claim, en een plek waar je nog niet had gekeken ([05.1](05.1-relationships%20jeffrey.ipynb)). Dit notebook neemt één relatie met een mechanisme waar niemand aan twijfelt, en stelt de vraag die meestal wordt overgeslagen: **is zij stabiel genoeg om als model vast te leggen?**

De relatie: in de Nederlandse COVID-cijfers volgen sterfgevallen op positieve tests. Iemand die positief test en daaraan overlijdt, doet dat ongeveer twee weken later, dus de sterfgevallen die veertien dagen vanaf nu worden geregistreerd, zijn de overlijdens *van* de positieve tests van vandaag. Dat is een vertraging (*lag*), en zodra die vertraging is gecorrigeerd, zouden de twee tijdreeksen gelijk moeten opgaan — zolang de fractie van positieve tests die in overlijden eindigt tenminste gelijk blijft.

Die laatste voorwaarde is de claim, en het is exact het type aanname dat een rechte lijn maakt: één vaste verhouding van overlijdens tot tests, gelijk op elke dag. Dit notebook fit die lijn, toetst haar op de manier waarop [04.5](../lesson4/04.5-fitting-the-residual%20jeffrey.ipynb) elk model toetst — op het residu — en vindt een periode waarin de lijn het hele verhaal vertelt, én een periode waarin dat niet meer het geval is. De laatste sectie draagt het gebroken model aan jou over.

In [ ]:
import numpy as np
from goad_toolkit.analytics import DistributionFitter, fit_table
from goad_toolkit.config import DataConfig, FileConfig
from goad_toolkit.dataprocessor import CovidDataProcessor
from goad_toolkit.models import linear_model, mse, train_model
from goad_toolkit.visualizer import (
    ComparePlot,
    ComparePlotDate,
    FitPlotSettings,
    PlotFits,
    PlotSettings,
    ResidualPlot,
)
from loguru import logger

## 5.3.1 De data, en wat de pipeline besluit

Dagelijkse Nederlandse COVID-cijfers, oktober 2020 tot juni 2021: positieve tests en sterfgevallen per dag, één keer gedownload en gecachet. `CovidDataProcessor` is een `Pipeline` van het type dat les 1 bouwde, en elke stap erin is een inhoudelijke beslissing over wat de data *betekent*, in plaats van een simpel opschoonklusje:

1. `DiffValues` op `deaths` — de bron rapporteert het cumulatieve aantal; het verschil is het aantal overlijdens op die dag.
2. `ShiftValues(period=-14)` → `deaths_shifted` — de overlijdens die veertien dagen *later* zijn geregistreerd, worden teruggeplaatst naar de rij van vandaag, naast de tests die ertoe hebben geleid. Een model tussen beide heeft deze lag nodig; zonder lag hebben de paren niets met elkaar te maken.
3. `SelectDataRange` — het tijdvenster.
4. `RollingAvg(window=7)` op beide — registraties kennen een wekelijkse cyclus (minder meldingen in het weekend) die geen onderdeel is van het virus zelf.
5. `ZScaler` → `*_zscore` — overlijdens tellen in tientallen per dag, tests in duizenden. Om te zien of de twee reeksen dezelfde *vorm* hebben, brengen we ze op één schaal.

De lag is het cruciale element om vast te houden. Veertien dagen is op zichzelf een claim over het biologische mechanisme — besmetting, ziekte, overlijden — en `DataConfig(period=-14)` is waar die claim expliciet leeft.

In [ ]:
processor = CovidDataProcessor(FileConfig(), DataConfig())
data = processor.process()
print(f"{len(data)} days, {data.index.min().date()} to {data.index.max().date()}")
data[["positivetests", "deaths", "deaths_shifted", "positivetests_zscore", "deaths_shifted_zscore"]].head(3)

In [ ]:
fig, ax = ComparePlot(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="z-score",
                                   title="Positive tests and (lagged) deaths, on one scale")).plot(
    data=data, x="date", y1="positivetests_zscore", y2="deaths_shifted_zscore",
)

Lees het als twee helften. Gedurende de herfst en winter liggen de curven bovenop elkaar: wanneer het aantal tests stijgt, stijgen de overlijdens twee weken later met een evenredige hoeveelheid, en wanneer het aantal tests daalt, dalen ook de overlijdens. Vanaf maart scheiden hun wegen zich — de tests klimmen opnieuw en de overlijdens volgen niet meer. Houd dat in gedachten, maar fit er nog niet op. De eerlijke manier om te achterhalen of één vaste verhouding deze relatie beschrijft, is door de lijn te fitten waar ze overduidelijk standhoudt, en vervolgens aan het residu te vragen wat er overblijft.

## 5.3.2 Een lijn, op het venster waar de verhouding constant lijkt

Het Nederlandse vaccinatieprogramma startte op 6 januari 2021. Fit de lijn op de dagen ervóór: `deaths_shifted ≈ a · positivetests + b`, met goad's modelleergereedschap — `linear_model` is de vorm, `mse` is het verlies, `train_model` vindt `a` en `b`. De grenzen (*bounds*) drukken uit wat je al weet: een ratio van overlijdens tot tests kan niet negatief zijn, en het startpunt (intercept) evenmin.

In [ ]:
VACCINATION_START = "2021-01-06"
before = data.index < VACCINATION_START

tests = data["positivetests"].to_numpy()
deaths = data["deaths_shifted"].to_numpy()

params_before = train_model(tests[before], deaths[before], linear_model, mse, [0.01, 1.0],
                            bounds=[(0, 1.0), (0, None)])
print(f"deaths ≈ {params_before[0]:.4f} · tests + {params_before[1]:.1f}")

early = data[before].copy()
early["predicted"] = linear_model(tests[before], params_before)
early["residual"] = early["deaths_shifted"] - early["predicted"]

fig, ax = ComparePlot(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="deaths",
                                   title="Before vaccination: deaths as a straight line in positive tests")).plot(
    data=early, x="date", y1="deaths_shifted", y2="predicted",
)

De lijn volgt de wintergolf omhoog, over de piek heen en weer naar beneden. Een fit die er goed uitziet is waar de meeste analyses stoppen, en het is exact de verkeerde plek om te stoppen: de grafiek toont wat het model wél heeft gepakt, en de echte vraag is wat het heeft gemist. Trek de lijn af en bekijk de fout, dag na dag.

In [ ]:
fig, ax = ResidualPlot(PlotSettings(figsize=(11, 3.5), xlabel="date", ylabel="error",  # ty: ignore[invalid-argument-type]
                                    title="Before vaccination: what the line did not explain")).plot(
    data=early, x="date", y="residual", date=VACCINATION_START, datelabel="vaccination starts",
    interval=2,
)

Golven van een week of twee aan weerszijden van nul — de omvang die een 7-daagse afvlakking van ruizige dagtellingen achterlaat — en geen drift over het venster. Dat is de eerste controle op een residu: **geen structuur in de tijd**, en daar slaagt het model voor. De tweede controle is de verdeling van de fout zelf, en dat is waar `DistributionFitter` voor dient binnen een modelleerlus: fit elke continue familie en lees de tabel af op consensus, niet enkel op een toevallige winnaar.

In [ ]:
fitter = DistributionFitter(seed=42)
fits_before = fitter.fit(early["residual"].to_numpy(), discrete=False)
fit_table(fits_before)[["distribution", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]].head(6)

In [ ]:
fig = PlotFits(PlotSettings(figsize=(12, 4), xlabel="error", ylabel="density",
                            title="Before vaccination: the residual, top three fits")).plot(
    data=early["residual"].to_numpy(), fit_results=fits_before,
    fitplotsettings=FitPlotSettings(bins=25, max_fits=3),
)

Gecentreerd rond nul, en een half dozijn families binnen een paar log-likelihood punten van elkaar — beta, weibull, gamma, lognormal, skew-normal — en geen van alle verworpen door KS. De enige familie die *wél* wordt verworpen, `uniform`, wordt verworpen om de juiste reden: de fouten clusteren rond nul in plaats van gelijkmatig te spreiden. Wanneer de winnaar niet betekenisvol voorliegt, kan de data de families niet van elkaar onderscheiden — en dát is exact hoe zuivere ruis eruitziet.

Zeg het expliciet, want dit is de lat waarlangs de rest van dit notebook elk model legt: **een symmetrisch residu uit een plausibele familie, zonder patroon in de tijd, is hoe "dit model is klaar" eruitziet.** Geen perfecte fit — de wekelijkse golven zijn er nog steeds — maar er blijft niets over dat een *vorm* heeft, en daardoor niets dat wijst op een mechanisme dat het model nog mist. Op dit venster is één vaste ratio van overlijdens tot tests nagenoeg het hele verhaal.

## 5.3.3 Dezelfde lijn, op het volledige venster

Neem nu het model exact zoals gefit — dezelfde `a`, dezelfde `b`, niets opnieuw geschat — en pas het toe op elke dag in het totale venster. Niets in het model weet welke maand het is, dus als de ratio daadwerkelijk één vast getal is, zou het residu eruit moeten blijven zien zoals in de bovenstaande plot.

In [ ]:
data["predicted"] = linear_model(tests, params_before)
data["residual"] = data["deaths_shifted"] - data["predicted"]
print(f"mse before vaccination {mse(deaths[before], data.loc[before, 'predicted']):.0f}, "
      f"after {mse(deaths[~before], data.loc[~before, 'predicted']):.0f}")

fig, ax = ComparePlotDate(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="deaths",
                                       title="The pre-vaccination line, run over the full window")).plot(
    data=data, x="date", y1="deaths_shifted", y2="predicted",
    date=VACCINATION_START, datelabel="vaccination starts",
)

In [ ]:
fig, ax = ResidualPlot(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="error",
                                    title="The full window: a residual with a shape")).plot(
    data=data, x="date", y="residual", date=VACCINATION_START,
    datelabel="vaccination starts", interval=1,
)

In [ ]:
fits_full = fitter.fit(data["residual"].to_numpy(), discrete=False)
fit_table(fits_full)[["distribution", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]].head(4)

Laat de plot het argument maken. Links van de lijn is het residu wat je zojuist al accepteerde als ruis. Rechts ervan glijden de fouten onder nul en blijven daar: vanaf maart, maand na maand, voorspelt de lijn meer overlijdens dan er in werkelijkheid plaatsvinden, en de reeks herstelt zich nooit meer. De fit-tabel vertelt hetzelfde in haar eigen taal — elke familie wordt verworpen, en `uniform` "wint", wat een fitter rapporteert wanneer de fouten uitgesmeerd liggen over een trend in plaats van geclusterd rond een centrum.

Dat is geen verdelingsprobleem, en het is ook geen fit-probleem: dezelfde lijn was een maand eerder nog pure ruis. **De relatie is fundamenteel veranderd.** Na januari leidt hetzelfde aantal positieve tests tot minder overlijdens, en een model met één vaste ratio heeft geen enkele manier om dat uit te drukken. Het residu deed dat zojuist wél — en het gaf ook grofweg aan wanneer.

## 5.3.4 Jouw beurt: benoem de vorm, voeg de term toe

Twee reacties op zo'n residu liggen voor de hand, en beide zijn fout.

De eerste is om het model veel complexer te maken — meer parameters, een tijdspolynoom, een aparte lijn per maand — totdat het residu met brute kracht plat wordt gedrukt. Dat matcht het plaatje, verklaart helemaal niets, en les 6 laat zien wat dat aanricht bij voorspellingen.

De tweede is om vaccinatie maar weg te laten uit het verhaal: de verhouding veranderde, de redenen zijn complex, rapporteer de correlatie en ga verder. Dat gooit het enige weg wat het residu je net aanreikte: *wanneer* de verandering begon.

Het probleem is prima op te lossen als je iets langer nadenkt over wat de data betekent. Je weet drie dingen: vaccinatie begon op 6 januari 2021. De uitrol begon met de mensen met het hoogste overlijdensrisico. De fractie van positieve tests die eindigt in overlijden moet dus *dalen* naarmate de uitrol vordert — niet van de ene op de andere dag, maar over de weken die nodig zijn om hen te bereiken.

De vraag is dus niet "welk model?". De vraag is: **wat doet de verhouding van overlijdens tot tests in de tijd?** Eén waarde, dan een overgang, dan een andere waarde — welke vorm is dat? Kijk opnieuw naar het residu, of bereken de ratio direct en plot hem. Schrijf vervolgens die vorm als term in het model, naast de lijn die je al hebt, met het dagnummer als input. `goad_toolkit.models.logistic` is een van de functies die goad meelevert; of dit de juiste is, en hoe hij gecombineerd moet worden met de lijn — opgeteld, of ermee vermenigvuldigd — is de beslissing die het model *definieert*. Beslis dat op basis van wat de claim betekent, niet op basis van welke versie toevallig beter fit.

Drie dingen om vast te stellen vóórdat je ook maar iets traint:

1. **De vorm**, in één zin over de ratio, vóórdat je een functie aanraakt.
2. **Hoe hij combineert met de lijn.** "Een vast aantal overlijdens per dag verdwijnt" en "de ratio van overlijdens tot tests daalt" zijn twee wezenlijk verschillende claims, en het zijn twee verschillende formules.
3. **Startwaarden en grenzen (bounds).** `a` en `b` heb je al. Voor de term die je toevoegt, vertelt de z-score plot grofweg wanneer de omslag plaatsvindt en welke kant hij opgaat; een optimizer die mijlenver van het antwoord start, vindt een ander optimum, en bounds zijn hoe je vastlegt wat een parameter mag betekenen.

Werk in de onderstaande cellen. De lus is al aangesloten, dus zodra `covid_model` iets doet, draait de rest vanzelf. Het uitgewerkte antwoord staat in [05.4](05.4-covid-solution%20jeffrey.ipynb) — lees dat nadat je eigen residu plat is, niet ervoor.

In [ ]:
# >>> Your turn: look at the ratio itself. What does deaths_shifted / positivetests do over time? <<<

### Model

De modelfunctie neemt twee inputs per dag — de positieve tests en het dagnummer, samengevoegd in één array — en een lijst van parameters. De onderstaande versie is de rechte lijn en niets anders: hij gebruikt de eerste twee parameters en negeert de rest, wat exact het model is dat zojuist faalde. Vervang dit door jouw model, en geef `initial_params` en `bounds` een invoer voor elke parameter die je toevoegt.

In [ ]:
day = np.arange(len(data)).astype(float)
X = np.stack([tests, day], axis=1)
vaccination_day = float(np.argmax(data.index >= VACCINATION_START))
print(f"X has shape {X.shape}: tests in column 0, day number in column 1; vaccination starts on day {vaccination_day:.0f}")


def covid_model(X: np.ndarray, params: list[float]) -> np.ndarray:
    """>>> Your turn: the line, combined with the shape you named. <<<"""
    a, b = params[:2]
    return linear_model(X[:, 0], [a, b])


initial_params = [params_before[0], params_before[1]]  # >>> add starting values for your own term <<<
bounds = [(0, 1.0), (0, None)]  # >>> and what each parameter is allowed to mean <<<

### Train

`train_model` minimaliseert het verlies over de parameters, vanaf jouw startwaarden, binnen jouw grenzen. Mislukt dit, dan is de gebruikelijke reden dat `params` een andere lengte heeft dan wat `covid_model` uitpakt.

In [ ]:
try:
    params = train_model(X, deaths, covid_model, mse, initial_params, bounds=bounds)
    logger.success(f"fitted parameters: {np.round(params, 4)}")
except Exception as error:
    logger.error(f"training failed: {error}")
    params = initial_params

### Voorspel

De fit tegen de data, met de start van de vaccinatie gemarkeerd — en vervolgens het residu, wat de plot is die beslist of je klaar bent.

In [ ]:
data["your model"] = covid_model(X, params)
data["your residual"] = data["deaths_shifted"] - data["your model"]
print(f"mse: the pre-vaccination line on the full window {mse(deaths, data['predicted']):.0f}  →  "
      f"your model {mse(deaths, data['your model']):.0f}")

fig, ax = ComparePlotDate(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="deaths",
                                       title="Deaths, and your model")).plot(
    data=data, x="date", y1="deaths_shifted", y2="your model",
    date=VACCINATION_START, datelabel="vaccination starts",
)

In [ ]:
fig, ax = ResidualPlot(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="error",
                                    title="What your model did not explain")).plot(
    data=data, x="date", y="your residual", date=VACCINATION_START,
    datelabel="vaccination starts", interval=1,
)

### Toets het residu

Dezelfde twee controles als in §5.3.2, langs dezelfde lat: geen structuur over de tijd, en een symmetrische verdeling die door meerdere families nagenoeg even goed wordt gefit. Als de lange negatieve reeks er nog steeds zit, zegt de term die je toevoegde niet wat je bedoelde — ga terug naar de vorm, niet naar de optimizer.

In [ ]:
fits_yours = fitter.fit(data["your residual"].to_numpy(), discrete=False)
print(fit_table(fits_yours)[["distribution", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]].head(4).to_string())

fig = PlotFits(PlotSettings(figsize=(12, 4), xlabel="error", ylabel="density",
                            title="Your residual, top three fits")).plot(
    data=data["your residual"].to_numpy(), fit_results=fits_yours,
    fitplotsettings=FitPlotSettings(bins=30, max_fits=3),
)

## 5.3.5 Wat op te schrijven

1. **De vorm, in woorden, en de formule die het werd.** Waarom die combinatie met de lijn en niet de andere — de zin over de ratio is de rechtvaardiging.
2. **Wat de gefitte parameters vertellen.** Waar de omslag halverwege is, hoe steil hij is, en hoe dat zich verhoudt tot de vaccinatie-uitrol. Die vergelijking is de bevinding.
3. **Wat het model niet zegt.** Het heeft een term op de kalender, geen term voor vaccinatie zelf. Het einde van de wintergolf, wie er werd getest, en de volgorde waarin leeftijdsgroepen werden gevaccineerd vallen allemaal op dezelfde kalender. Welke van 05.1's drie pijlers levert de fit, en welke is nog steeds een claim die je op een andere manier zou moeten verdedigen?
4. **Of je klaar bent**, volgens de lat die §5.3.2 neerlegde — en wat het residu nog bevat dat een serieus model hiervan zou moeten behandelen.